# TODO: You must set checkpoint_directory to where you want to save and load the model

In [1]:
checkpoint_base_directory ="/Users/jameswalker/Programming/Checkpoints/MappoCheckpoint"
checkpoint_number = 10
# checkpoint_directory = "/content/drive/MyDrive/rllib_checkpoints"

In [9]:
import ray 

ray.shutdown()
ray.init() 

2025-04-26 21:15:51,635	INFO worker.py:1852 -- Started a local Ray instance.


Python version:,3.9.17
Ray version:,2.44.1


In [10]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from tqdm import tqdm 
from metadrive.envs.rllib_delegator_env import RLLibDelegatorEnv 
from ray.rllib.core.rl_module.multi_rl_module import MultiRLModuleSpec
from ray.rllib.core.rl_module.rl_module import RLModuleSpec

# Not sure if sgd_minibatch_size is being handled properly, but train_batch_size and worker_num are for sure
# worker_num = 1
# train_batch_size = int(1024)
# sgd_minibatch_size = max(256, int(train_batch_size/10))

config = ( PPOConfig()
    .environment(
        RLLibDelegatorEnv,
        disable_env_checking=True
        )
    .env_runners(
        num_env_runners=3, 
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0
        )
    .multi_agent(
        policies={"agent0", "agent1"},
        policy_mapping_fn=lambda agent_id, episode, **kw: agent_id,
    )
    .framework('torch')
    # The below .learners comes from the migration guide when they talk about no GPU but multi-CPU
    .learners(
        num_learners=2,
        num_cpus_per_learner=2,
        # num_gpus_per_learner=0,
    )
    .training(
        # train_batch_size=train_batch_size, # Deprecated
        train_batch_size_per_learner=int(2048), # Replacement for train_batch_size
        # sgd_minibatch_size=sgd_minibatch_size, # This got deprecated
        minibatch_size=int(256), # This might be the replacement
        lr=1e-4,
        entropy_coeff=0.001,
    )    
    .rl_module(
        # Use a non-default 32,32-stack with ReLU activations.
        # This is for PPO since it uses multi-layer perceptrons (MLP)
        model_config=DefaultModelConfig(
            fcnet_hiddens=[32, 32],
            fcnet_activation="relu",
        ),
        rl_module_spec=MultiRLModuleSpec(
            rl_module_specs={
                "agent0": RLModuleSpec(),
                "agent1": RLModuleSpec(),
            }
        ),
    )
    # The below, commented .training illustrates grid search for learning rate,
    # comes from https://github.com/ray-project/ray/blob/master/rllib/examples/gpus/fractional_gpus_per_learner.py
    # .training(
    #     lr=tune.grid_search([0.005, 0.003, 0.001, 0.0001])
    # )
    # .evaluation(
    #     # TODO evaluation_interval?
    #     # evaluation_interval=1,
    #     evaluation_num_env_runners=1, 
    #     # evaluation_duration=10000,
    #     # WARNING algorithm_config.py:4593 -- You have specified 1 evaluation workers, but your `evaluation_interval` is 0 or None! Therefore, evaluation doesn't occur automatically with each call to `Algorithm.train()`. Instead, you have to call `Algorithm.evaluate()` manually in order to trigger an evaluation run.
    # )
    # .resources is only for setting num_cpus_for_main_process, I don't think we need to mess with it
    # .resources(
    #     num_gpus=int(0),  # This is deprecated
    #     num_cpus_for_main_process=int(1), # Defaults to 1
    # )  
    .debugging(log_level='INFO') # INFO, DEBUG, ERROR, WARN
)



In [11]:
# config.build() is deprecated, use build_algo() instead
algo = config.build_algo()
# print(f"Type of algo: {type(algo)}")


/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/algorithms/algorithm.py:512: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and m

(_WrappedExecutable pid=32892) 2025-04-26 21:16:07,384	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!


# TRAIN THE ALGORITHM
100 training iterations took about 2 minutes on my Mac with M1 chip
If setting it up on a server, just try 100 iterations and see if you can save and load the Algorithm instance. After that try 10,000 iterations

In [4]:
# from collections import deque
# SMOOTHING_WINDOW = 10  
# returns_window = deque(maxlen=SMOOTHING_WINDOW)

iteration_num = 10_000
ENV_RUNNER_RESULTS = "env_runners"
EPISODE_RETURN_MEAN = "episode_return_mean"
EVALUATION_RESULTS = "evaluation"
for iteration in tqdm(range(0, iteration_num)):
    results = algo.train()
    # # train_R = results["env_runners"].get("episode_return_mean", float("nan"))
    # # returns_window.append(train_R)

    # # # compute the smoothed average
    # # smooth_R = sum(returns_window) / len(returns_window)

    # # print(f"it={iteration:4d}   R_train={train_R:6.2f}   R_smooth={smooth_R:6.2f}", end="")

    # # # if you’ve enabled built-in evaluation in your config:
    # # if "evaluation" in results and "env_runners" in results["evaluation"]:
    # #     eval_R = results["evaluation"]["env_runners"]["episode_return_mean"]
    # #     print(f"   R_eval={eval_R:6.2f}", end="")

    # print()
    if iteration % 50 == 0:
        # Save Algorithm checkpoint.
        checkpoint_directory = checkpoint_base_directory + str(checkpoint_number)
        algo.save_to_path(checkpoint_directory)
        checkpoint_number = checkpoint_number + 1
        print(f"Saved to place")
        if ENV_RUNNER_RESULTS in results:
            mean_return = results["env_runners"].get("episode_return_mean", float("nan"))
            print(f"iter={iteration} R={mean_return}", end="")
            if (EVALUATION_RESULTS in results and ENV_RUNNER_RESULTS in results[EVALUATION_RESULTS]):
                Reval = results[EVALUATION_RESULTS][ENV_RUNNER_RESULTS][EPISODE_RETURN_MEAN]
                print(f" R(eval)={Reval}", end="")
            print()
    # For seeing all the metrics we can use in the result dict():
    # for key in result['env_runners'].keys():
    #     print(f"Key {key}: {result['env_runners'][key]}")
    # # TODO Are these good metrics below? Not sure exactly what either means
    # if iteration % (iteration_num/10) == 0:
        # print(f"Iteration {iteration} Episode Length Mean: {result['env_runners']['episode_len_mean']}")
        # print(f"Iteration {iteration} Agent Episode Returns Mean: {result['env_runners']['agent_episode_returns_mean']}")


NameError: name 'tqdm' is not defined

# Save Algorithm instance
"An Algorithm instance typically holds a neural network for computing actions, called the policy, the RL environment you want to optimize against, a loss function, an optimizer, and some code describing the algorithm’s execution logic, like determining when to collect samples, when to update your model, etc."

We can checkpoint an Algorithm to save its state then load it into a new Algorithm instance at a later date.

In [6]:
# Save Algorithm checkpoint.
algo.save_to_path(checkpoint_directory)

# Display the Algorithm RLModule to check we loaded the same RLModule later on
print(type(algo))
algo.get_module()

<class 'ray.rllib.algorithms.ppo.ppo.PPO'>


# How to debug AssertionError
AssertionError: Can not call this API after engine initialization!

Run the cell below. It should fix this error. This error gets raised if you press 'ESC' and shutdown the Environment but the renderer doesn't properly shutdown.

In [7]:
env.close()

# Testing your trained model

In [ ]:
import torch
import numpy as np
from metadrive.envs.marl_envs.rllib_mappo_env import RLLibMappoEnv
import gymnasium as gym
from ray.rllib.core.rl_module.rl_module import RLModule
import os

checkpoint_directory = checkpoint_base_directory + str(checkpoint_number)

agent0_rl_module = RLModule.from_checkpoint(
        os.path.join(
            checkpoint_directory,
            "learner_group",
            "learner",
            "rl_module",
            "agent0",
        )
    )
agent1_rl_module = RLModule.from_checkpoint(
        os.path.join(
            checkpoint_directory,
            "learner_group",
            "learner",
            "rl_module",
            "agent1",
        )
    )

# modules that were created with RLModule.from_checkpoint(...)
agent_modules = {
    "agent0": agent0_rl_module,
    "agent1": agent1_rl_module,
}
action_dist_cls = {
    aid: mod.get_inference_action_dist_cls() for aid, mod in agent_modules.items()
}
print("========== Printing action_dist_cls ===========")
for action_dist in action_dist_cls.keys():
    action_dist_class = action_dist_cls[action_dist]
    # print("Sampled: ", action_dist_cls[action_dist].sample(sample_shape=(2,)))

env = RLLibMappoEnv(
    {
        # "start_seed": 0,
        "use_render": True,
        # "manual_control": True,
    }
)
obs, info = env.reset()

episode_cost = {"agent0": 0, "agent1": 0}

try:
    for step in range(1, 10_000):
        if step % 100 == 0:
            print(f"Iteration {step}")

        # build a separate forward-pass dict for every agent
        actions = {}
        with torch.no_grad():
            for aid, ob in obs.items():
                # ChatGPT says this way is faster/better
                fwd_ins = {"obs": torch.as_tensor(ob).unsqueeze(0)}
                out = agent_modules[aid].forward_exploration(fwd_ins)
                dist = action_dist_cls[aid].from_logits(out["action_dist_inputs"])
                # dist.to_deterministic()
                actions[aid] = dist.sample()[0].cpu().numpy()
                # actions[aid] = dist.sample()

                # Compare with old
                # old_fwd_ins = {"obs": torch.Tensor([ob])}
                # old_fwd_outputs = agent_modules[aid].forward_exploration(old_fwd_ins)
                # old_action_dist = action_dist_cls[aid].from_logits(old_fwd_outputs["action_dist_inputs"])
                # old_action = old_action_dist.sample()[0].numpy()
                # actions[aid] = old_action

        obs, reward, terminated, truncated, info = env.step(actions)

        if env.config["use_render"]:
            # Render with race information
            env.render(
                text={
                    "Race Progress": {agent_id: f"{progress:.2f}" for agent_id, progress in env.agent_progress.items()},
                    "Leading": next((agent_id for agent_id in env.agents.keys() if env.agent_progress.get(agent_id, 0) == max(env.agent_progress.values())), None),
                    "Race Winner": env.race_winner if env.race_finished else "None",
                }
            )

        # Check if all agents are done
        if all(terminated.values()):
            print("All agents terminated. Resetting environment.")
            obs, _ = env.reset()
finally:
    env.close()
print("Finished")

[INFO] Environment: RLLibMappoEnv


[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector(), main_camera: MainCamera(1200, 900), dashboard: DashBoard()]
[INFO] Render Mode: onscreen
[INFO] Horizon (Max steps per agent): 2000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: CocoaGraphicsPipe


========== Printing action_dist_cls ===========


[INFO] Start Scenario Index: 0, Num Scenarios : 100
[WARNING] env.vehicles will be deprecated soon. Use env.agents instead (base_env.py:735)


Created 5 checkpoints along the track
Found ending block: 5r
Created finish line across the entire track at the ending segment


RuntimeError: Boolean value of Tensor with more than one value is ambiguous